In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
                              RocCurveDisplay, classification_report)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42


In [ ]:
# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = malignant, 1 = benign (sklearn's internal encoding)

# Human-readable diagnosis column, matching UCI's 'M'/'B' convention
df['diagnosis'] = df['target'].map({0: 'M', 1: 'B'})

# Encode class labels explicitly with LabelEncoder (as required by the lab steps)
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['diagnosis'])  # B -> 0, M -> 1 alphabetically

print("Dataset shape:", df.shape)
print("Classes:", dict(zip(le.classes_, le.transform(le.classes_))))
df.head()


In [ ]:
class_counts = df['diagnosis'].value_counts()
print(class_counts)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='diagnosis', data=df, palette='Set2', ax=ax[0])
ax[0].set_title('Class Distribution (Count)')
ax[0].set_xlabel('Diagnosis (M = Malignant, B = Benign)')

ax[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%',
          colors=sns.color_palette('Set2'), startangle=90)
ax[1].set_title('Class Distribution (%)')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(18, 14))
corr = df[data.feature_names].corr()
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, cbar_kws={'shrink': 0.7})
plt.title('Feature Correlation Heatmap (30 Features)')
plt.tight_layout()
plt.show()

# Top features most correlated with the target
target_corr = df[list(data.feature_names) + ['target_encoded']].corr()['target_encoded'].drop('target_encoded')
target_corr_sorted = target_corr.abs().sort_values(ascending=False)
print("Top 10 features most correlated with diagnosis:")
print(target_corr_sorted.head(10))

plt.figure(figsize=(8, 6))
target_corr_sorted.head(10).sort_values().plot(kind='barh', color='teal')
plt.title('Top 10 Features Correlated with Diagnosis')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()


In [ ]:
X = df[data.feature_names]
y = df['target_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
print("\nTraining class distribution:\n", y_train.value_counts(normalize=True))
print("\nTesting class distribution:\n", y_test.value_counts(normalize=True))


In [ ]:
dt_baseline = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_baseline.fit(X_train, y_train)

y_pred_baseline = dt_baseline.predict(X_test)
print("Baseline Decision Tree Test Accuracy: {:.4f}".format(accuracy_score(y_test, y_pred_baseline)))
print("Baseline Tree Depth:", dt_baseline.get_depth())
print("Baseline Number of Leaves:", dt_baseline.get_n_leaves())


In [ ]:
dt_param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=dt_param_grid,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=False
)

dt_grid.fit(X_train, y_train)

print("Best Decision Tree Params:", dt_grid.best_params_)
print("Best CV Accuracy: {:.4f}".format(dt_grid.best_score_))


In [ ]:
# Table 1: Decision Tree Hyperparameter Evaluation using 5-Fold Cross-Validation
# Summarized by criterion & max_depth (averaged over min_samples_split / min_samples_leaf)
dt_results = pd.DataFrame(dt_grid.cv_results_)

dt_results['f1_mean'] = np.nan  # placeholder, computed below via cross_val_score for the summary table

summary_rows = []
for criterion in dt_param_grid['criterion']:
    for depth in dt_param_grid['max_depth']:
        subset = dt_results[
            (dt_results['param_criterion'] == criterion) &
            (dt_results['param_max_depth'] == depth)
        ]
        avg_acc = subset['mean_test_score'].mean() * 100

        # Compute average F1 across the same subset of configurations via cross_val_score
        f1_scores = []
        for _, row in subset.iterrows():
            clf = DecisionTreeClassifier(
                criterion=criterion, max_depth=depth,
                min_samples_split=row['param_min_samples_split'],
                min_samples_leaf=row['param_min_samples_leaf'],
                random_state=RANDOM_STATE
            )
            f1 = cross_val_score(clf, X_train, y_train, cv=cv_strategy, scoring='f1').mean()
            f1_scores.append(f1)

        summary_rows.append({
            'Criterion': criterion,
            'Max Depth': 'None' if depth is None else depth,
            'Avg CV Accuracy (%)': round(avg_acc, 2),
            'Avg CV F1 Score': round(np.mean(f1_scores), 4)
        })

table1_dt = pd.DataFrame(summary_rows)
table1_dt


In [ ]:
best_dt = dt_grid.best_estimator_
print("Selected Decision Tree Hyperparameters:")
for k, v in dt_grid.best_params_.items():
    print(f"  {k}: {v}")

# Visualize top few levels of the best tree
plt.figure(figsize=(20, 10))
plot_tree(best_dt, max_depth=3, feature_names=data.feature_names,
          class_names=['Benign', 'Malignant'], filled=True, fontsize=8)
plt.title('Best Decision Tree (Top 3 Levels)')
plt.show()


In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=rf_param_grid,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=False
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest Params:", rf_grid.best_params_)
print("Best CV Accuracy: {:.4f}".format(rf_grid.best_score_))


In [ ]:
# Table 2: Random Forest Hyperparameter Evaluation using 5-Fold Cross-Validation
# Summarized by n_estimators, max_depth & max_features (averaged over bootstrap)
rf_results = pd.DataFrame(rf_grid.cv_results_)

summary_rows_rf = []
for n_est in rf_param_grid['n_estimators']:
    for depth in rf_param_grid['max_depth']:
        for max_feat in rf_param_grid['max_features']:
            subset = rf_results[
                (rf_results['param_n_estimators'] == n_est) &
                (rf_results['param_max_depth'] == depth) &
                (rf_results['param_max_features'] == max_feat)
            ]
            avg_acc = subset['mean_test_score'].mean() * 100

            f1_scores = []
            for _, row in subset.iterrows():
                clf = RandomForestClassifier(
                    n_estimators=n_est, max_depth=depth, max_features=max_feat,
                    bootstrap=row['param_bootstrap'], random_state=RANDOM_STATE, n_jobs=-1
                )
                f1 = cross_val_score(clf, X_train, y_train, cv=cv_strategy, scoring='f1').mean()
                f1_scores.append(f1)

            summary_rows_rf.append({
                'n_estimators': n_est,
                'Max Depth': 'None' if depth is None else depth,
                'Max Features': max_feat,
                'Avg CV Accuracy (%)': round(avg_acc, 2),
                'Avg CV F1 Score': round(np.mean(f1_scores), 4)
            })

table2_rf = pd.DataFrame(summary_rows_rf)
table2_rf


In [ ]:
best_rf = rf_grid.best_estimator_
print("Selected Random Forest Hyperparameters:")
for k, v in rf_grid.best_params_.items():
    print(f"  {k}: {v}")


In [ ]:
dt_fold_scores = cross_val_score(best_dt, X_train, y_train, cv=cv_strategy, scoring='accuracy')
rf_fold_scores = cross_val_score(best_rf, X_train, y_train, cv=cv_strategy, scoring='accuracy')

table3 = pd.DataFrame({
    'Fold 1': [dt_fold_scores[0], rf_fold_scores[0]],
    'Fold 2': [dt_fold_scores[1], rf_fold_scores[1]],
    'Fold 3': [dt_fold_scores[2], rf_fold_scores[2]],
    'Fold 4': [dt_fold_scores[3], rf_fold_scores[3]],
    'Fold 5': [dt_fold_scores[4], rf_fold_scores[4]],
}, index=['Decision Tree', 'Random Forest'])

table3['Average'] = table3.mean(axis=1)
table3 = table3.round(4)
table3


In [ ]:
table3.T.plot(kind='line', marker='o', figsize=(9, 5))
plt.title('5-Fold Cross-Validation Accuracy: Decision Tree vs Random Forest')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.ylim(0.85, 1.0)
plt.legend(title='Model')
plt.tight_layout()
plt.show()


In [ ]:
models = {'Decision Tree': best_dt, 'Random Forest': best_rf}
metrics_summary = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    metrics_summary.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })

metrics_df = pd.DataFrame(metrics_summary).set_index('Model').round(4)
metrics_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign', 'Malignant'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name} — Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 6))
for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve: Decision Tree vs Random Forest')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
print("Decision Tree Classification Report")
print(classification_report(y_test, best_dt.predict(X_test), target_names=['Benign', 'Malignant']))

print(" Random Forest Classification Report ")
print(classification_report(y_test, best_rf.predict(X_test), target_names=['Benign', 'Malignant']))


In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=data.feature_names).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
importances.head(15).sort_values().plot(kind='barh', color='darkorange')
plt.title('Top 15 Feature Importances — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


In [ ]:
depths = range(1, 16)
train_scores, cv_scores = [], []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    cv_scores.append(cross_val_score(clf, X_train, y_train, cv=cv_strategy, scoring='accuracy').mean())

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, marker='o', label='Training Accuracy')
plt.plot(depths, cv_scores, marker='s', label='5-Fold CV Accuracy')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Training vs Cross-Validation Accuracy by Depth')
plt.legend()
plt.tight_layout()
plt.show()
